In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path)


In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Traget distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('deivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis = 1)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
df_clean = df.copy()
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(0)
df_clean = df_clean.dropna(subset=['Weather', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Delivery_Time'])


In [ ]:
df_clean.isnull().sum()

In [ ]:
#categorical columns
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

In [ ]:
df_clean.info()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 6: Write your code here:

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop(['Delivery_Time'], axis =1)
y = df_clean['Delivery_Time']

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X):
    X_fold_train, X_fold_val = X.values[train_idx], X.values[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")
print(f"RMSE: {rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_fold_pred, bins=50, edgecolor='black')
plt.title('Delivery time')
plt.xlabel('predictied time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
%pip install catboost

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
# Task Bonus: Write your code here:

In [ ]:
# TODO: Define models with hyperparameters of your choice
models = {
  "Random Forest": RandomForestRegressor(n_estimators=100,
    max_depth=20, random_state=42, n_jobs=-1),
  "CatBoost": CatBoostRegressor(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

In [ ]:
results = {}

for model_name in models:
  results[model_name] = {'MAE': [], 'MSE': []}

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kfold.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)  #

    results[model_name]['MAE'].append(mae)
    results[model_name]['MSE'].append(mse)